# Week 5 — Track A: RAG by hand

**SWE 485 · Advanced Software Engineering with AI · King Saud University**

You are going to build the retrieve → augment → generate loop yourself, over the
`minibank/` corpus, with every similarity score printed. Then you are going to
break it five times on purpose.

The one rule: **the cosine calculation must be yours.** Do not import a
similarity helper. The whole point of this lab is that "the tool found the
relevant code" reduces to a dot product and two magnitudes, and you only get
that by writing the line.

---


## Setup

Your API key must be stored as the Colab Secret `GEMINI_API_KEY`, not pasted
into a cell. Get one at [aistudio.google.com/apikey](https://aistudio.google.com/apikey).


In [ ]:
%pip install -U -q "google-genai" numpy

In [ ]:
import numpy as np
from google import genai
from google.genai import types

# In Colab: clone the seed repo first, then run this notebook from inside it.
#   !git clone <your-fork-url> && cd swe485-week05-rag-seed
import config

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:
    import os
    GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]

client = genai.Client(api_key=GEMINI_API_KEY)

print("embedding model :", config.EMBED_MODEL)
print("chat model      :", config.CHAT_MODEL)
print("offline mode    :", config.OFFLINE)

---

## A1 — Chunk

Chunking decides the **unit of retrieval**. Everything downstream inherits the
decision you make here, which is why the study material calls it the most
consequential design choice in the pipeline.

Read the next cell before you run it. You will be asked about it.


In [ ]:
import re, fnmatch, pathlib

def tokenize(text):
    """An approximation of the model's tokenizer: words and standalone symbols.

    It is not the real tokenizer, and it does not need to be. What it must be
    is DETERMINISTIC, so that every team's chunk boundaries land in the same
    place and the examiner can predict what you will see.
    """
    return re.findall(r"\w+|[^\w\s]", text)


def load_ragignore(root):
    """Read the exclusion list. Open the file. Actually read it."""
    patterns, f = [], root / ".ragignore"
    if f.exists():
        for line in f.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#"):
                patterns.append(line)
    return patterns


def is_ignored(rel, patterns):
    return any(
        fnmatch.fnmatch(rel, p) or fnmatch.fnmatch(rel, p.rstrip("/") + "/*")
        for p in patterns
    )


def chunk_file(path, rel, chunk_tokens, overlap_tokens):
    """Fixed-size chunking, aligned to line boundaries.

    Lines are accumulated until the token budget is spent. The next chunk backs
    up by up to `overlap_tokens` so that a cut point is not a total loss.
    """
    lines = path.read_text().splitlines()
    counts = [len(tokenize(l)) for l in lines]
    chunks, start = [], 0
    while start < len(lines):
        total, end = 0, start
        while end < len(lines) and total + counts[end] <= chunk_tokens:
            total += counts[end]
            end += 1
        if end == start:            # one line longer than the whole budget
            end = start + 1
        chunks.append({
            "id": f"{rel}#L{start+1}-L{end}",
            "path": rel, "start": start + 1, "end": end,
            "text": "\n".join(lines[start:end]), "tokens": total,
        })
        if end >= len(lines):
            break
        back, carried = end, 0
        while back > start + 1 and carried + counts[back-1] <= overlap_tokens:
            back -= 1
            carried += counts[back]
        start = back
    return chunks


def build_chunks(corpus_dir, chunk_tokens, overlap_tokens, suffixes):
    root = pathlib.Path(corpus_dir)
    patterns = load_ragignore(root)
    chunks, skipped = [], []
    for p in sorted(root.rglob("*")):
        if not p.is_file() or p.suffix not in suffixes:
            continue
        rel = str(p.relative_to(root))
        if is_ignored(rel, patterns) or p.name == "__init__.py":
            skipped.append(rel)
            continue
        chunks += chunk_file(p, rel, chunk_tokens, overlap_tokens)
    return chunks, skipped

In [ ]:
chunks, skipped = build_chunks(
    config.CORPUS_DIR, config.CHUNK_TOKENS, config.OVERLAP_TOKENS,
    config.INDEXABLE_SUFFIXES,
)

print(f"{len(chunks)} chunks indexed, {len(skipped)} files skipped\n")
for ch in chunks:
    print(f"  {ch['id']:36s} {ch['tokens']:4d} tokens")

print("\nNOT INDEXED:")
for s in skipped:
    print("  ", s)

**Stop and look at that output before going on.**

1. Which file produced **two** chunks instead of one, and where did the cut land?
   Open it and read the lines either side of the boundary. What ended up
   separated from what?
2. Which file is in the "NOT INDEXED" list for a reason that is not `__init__.py`?
   Find out why. The answer is in a file you have not opened yet.

Those two observations are failure modes 1 and 2, and you have just found both
of them before running a single query.


---

## A2 — Embed

Retrieval embeddings are **asymmetric**. A question and the passage that answers
it are not the same kind of text, and the API makes you say which is which:
`RETRIEVAL_DOCUMENT` for chunks, `CODE_RETRIEVAL_QUERY` for questions about code.

**Normalization.** `gemini-embedding-001` returns un-normalized vectors at any
dimensionality other than 3072. We ask for 768, so we must normalize ourselves.
This is section 3's transfer question in production: normalize at index time and
cosine collapses to a plain dot product.


In [ ]:
def normalize(v):
    v = np.asarray(v, dtype=np.float32)
    return v / np.linalg.norm(v)


def embed_documents(texts):
    result = client.models.embed_content(
        model=config.EMBED_MODEL,
        contents=[config.prepare_document(t) for t in texts],
        config=types.EmbedContentConfig(
            task_type=config.DOC_TASK_TYPE,
            output_dimensionality=config.EMBED_DIM,
        ),
    )
    vecs = np.stack([normalize(e.values) for e in result.embeddings])
    return vecs


if config.OFFLINE:
    cached = np.load("embeddings.npz", allow_pickle=True)
    doc_vectors = cached["vectors"]
    assert list(cached["ids"]) == [c["id"] for c in chunks], (
        "embeddings.npz does not match the current corpus or chunk settings. "
        "Re-run tools/build_embeddings.py."
    )
    print("loaded", doc_vectors.shape, "from embeddings.npz")
else:
    doc_vectors = embed_documents([c["text"] for c in chunks])
    print("embedded", doc_vectors.shape)

# Sanity check: normalized vectors have length 1.
print("norm of first vector:", float(np.linalg.norm(doc_vectors[0])))

---

## A3 — Retrieve  *(the graded step)*

Top-k is a k-nearest-neighbour search: embed the query, score every stored chunk
by cosine similarity, sort, take the top *k*. That is all it has ever been.

Because the document vectors are already normalized, `cosine()` below reduces to
`a @ b`. Write it out in full anyway the first time — you need to be able to say
what each term is.


In [ ]:
def cosine(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


def embed_query(query):
    result = client.models.embed_content(
        model=config.EMBED_MODEL,
        contents=[config.prepare_query(query)],
        config=types.EmbedContentConfig(
            task_type=config.QUERY_TASK_TYPE,
            output_dimensionality=config.EMBED_DIM,
        ),
    )
    return normalize(result.embeddings[0].values)


def retrieve(query, k=None, show_all=True):
    """Score every chunk, print the ranking, return the top k."""
    k = config.K if k is None else k
    qv = embed_query(query)
    scored = sorted(
        ((cosine(qv, doc_vectors[i]), chunks[i]) for i in range(len(chunks))),
        key=lambda pair: pair[0],
        reverse=True,
    )

    print(f"QUERY: {query}\n")
    shown = scored if show_all else scored[:k]
    for rank, (score, ch) in enumerate(shown, 1):
        marker = "<-- retrieved" if rank <= k else ""
        print(f"  {rank:2d}. {score:.4f}  {ch['id']:36s} {marker}")

    gap = scored[0][0] - scored[1][0]
    print(f"\n  margin between rank 1 and rank 2: {gap:.4f}")
    return [ch for _, ch in scored[:k]]

### Q0 — the control

This is the worked trace from the lecture, run for real. In section 3 you
hand-computed that a login query scored ≈0.99 against an auth chunk and ≈0.44
against a billing chunk, using two-dimensional toy vectors. These are 768
dimensions. Watch the same thing happen.


In [ ]:
top = retrieve("How does login work?")

---

## A4 — Generate, grounded and cited

Two things to notice in the prompt below. The model is told to answer **only**
from the context, and it is given an explicit way out: `NOT IN CONTEXT`.

That escape hatch is a **generation-side** mitigation, and Part B uses it to
separate the two kinds of fault. Keep it in mind.


In [ ]:
SYSTEM = (
    "You are answering questions about a codebase. "
    "Answer ONLY from the provided context. "
    "Cite the chunk id you used, in square brackets, after each claim. "
    "If the context does not contain the answer, reply exactly: NOT IN CONTEXT."
)


def build_context(retrieved):
    return "\n\n".join(
        f"--- chunk {ch['id']} ---\n{ch['text']}" for ch in retrieved
    )


def answer(query, k=None, system=SYSTEM, show_all=False):
    retrieved = retrieve(query, k=k, show_all=show_all)
    response = client.models.generate_content(
        model=config.CHAT_MODEL,
        contents=f"CONTEXT:\n{build_context(retrieved)}\n\nQUESTION: {query}",
        config=types.GenerateContentConfig(system_instruction=system),
    )
    print("\n" + "=" * 72)
    print(response.text)
    return response

In [ ]:
_ = answer("How does login work?")

---

# Part B — Break it

Five queries, five failure modes. For each one: run it, **read the score table
before you read the answer**, and decide whether what went wrong is a
**retrieval** fault or a **generation** fault.

You will be asked that question out loud, without notes, in the assessment.


### Q1 — bad chunking

`validate_token()` performs two independent checks: an expiry check and a
signature check. Find out which chunk each one landed in.


In [ ]:
_ = answer("How is the token signature verified?", show_all=True)

### Q2 — missed context

Before you blame the model, check what was indexed.


In [ ]:
_ = answer("How does the second factor get checked?", show_all=True)

### Q3 — stale source

Three files in this corpus have an opinion about how long a token lasts. They do
not agree. Look at which one won, and then work out which one is *right* — and
be precise about whether the fault is a stale **index** or a stale **source**,
because they are different faults with different fixes.


In [ ]:
_ = answer("How long does a session token last?", show_all=True)

### Q4 — wrong retrieval

Look at the margin between rank 1 and rank 2 on this one. Then ask yourself what
a larger `k` would and would not have fixed.


In [ ]:
_ = answer("What happens when a charge fails?", show_all=True)

### Q5 — needle in a haystack

This question needs three facts that live in three different files. Retrieval
will find some of them.


In [ ]:
_ = answer(
    "What happens to a user's active session when their subscription "
    "lapses mid-cycle?",
    show_all=True,
)

---

## The two-run demonstration

Take **one** query that failed above and run the generation step twice: once with
the `NOT IN CONTEXT` escape hatch, once without it.

Same retrieval. Same chunks. Opposite behaviour. That contrast is the clearest
thirty seconds in this lab, and it is worth having on screen when the examiner
asks you to classify the fault.


In [ ]:
NO_ESCAPE = (
    "You are answering questions about a codebase. "
    "Answer from the provided context. "
    "Cite the chunk id you used, in square brackets, after each claim."
)

QUERY = "How does the second factor get checked?"

print("### WITH the escape hatch " + "#" * 40)
_ = answer(QUERY, system=SYSTEM)

print("\n\n### WITHOUT the escape hatch " + "#" * 37)
_ = answer(QUERY, system=NO_ESCAPE)

---

# Part D — One improvement

Pick one failure, name the mode, apply the fix, show the before-and-after scores.

Below is a working AST-aware chunker, which is the fix for mode 1. It splits
Python along syntactic boundaries so that a function is never cut in half. Run
it, re-embed, and re-run Q1.

Note what it costs: it only works for Python, it needs a parser per language,
and it will happily emit a chunk far larger than your token budget if someone
writes a 900-line function. Fixed-size chunking has none of those problems. That
trade-off is the answer to "why doesn't everyone just do this?"


In [ ]:
import ast


def chunk_file_ast(path, rel, chunk_tokens, overlap_tokens):
    """Split Python on top-level definitions; fall back to fixed-size."""
    source = path.read_text()
    if path.suffix != ".py":
        return chunk_file(path, rel, chunk_tokens, overlap_tokens)

    try:
        tree = ast.parse(source)
    except SyntaxError:
        return chunk_file(path, rel, chunk_tokens, overlap_tokens)

    lines = source.splitlines()
    spans, cursor = [], 0
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            start = min(
                [d.lineno for d in node.decorator_list] + [node.lineno]
            ) - 1
            if start > cursor:
                spans.append((cursor, start))       # module-level preamble
            spans.append((start, node.end_lineno))
            cursor = node.end_lineno
    if cursor < len(lines):
        spans.append((cursor, len(lines)))

    out = []
    for start, end in spans:
        text = "\n".join(lines[start:end]).strip("\n")
        if not text.strip():
            continue
        out.append({
            "id": f"{rel}#L{start+1}-L{end}",
            "path": rel, "start": start + 1, "end": end,
            "text": text, "tokens": len(tokenize(text)),
        })
    return out


# Re-index with AST-aware chunking, then re-embed and re-run Q1.
import pathlib

_root = pathlib.Path(config.CORPUS_DIR)
_patterns = load_ragignore(_root)
chunks = []
for p in sorted(_root.rglob("*")):
    if not p.is_file() or p.suffix not in config.INDEXABLE_SUFFIXES:
        continue
    rel = str(p.relative_to(_root))
    if is_ignored(rel, _patterns) or p.name == "__init__.py":
        continue
    chunks += chunk_file_ast(p, rel, config.CHUNK_TOKENS, config.OVERLAP_TOKENS)

print(f"{len(chunks)} AST chunks\n")
for ch in chunks:
    print(f"  {ch['id']:40s} {ch['tokens']:4d} tokens")

doc_vectors = embed_documents([c["text"] for c in chunks])
print()
_ = answer("How is the token signature verified?", show_all=True)

---

## Before your assessment

You have five minutes and an examiner who will pick which of you presents.

- [ ] The notebook runs top to bottom without editing.
- [ ] A **score table** is visible on screen, not just an answer.
- [ ] One grounded answer cites the correct chunk.
- [ ] One Part B failure runs live and fails in public.
- [ ] Every member can answer, unscripted: *retrieval fault or generation fault,
      and what would fix it?*
- [ ] Outputs are saved in the committed notebook, as the examiner's fallback.
- [ ] **No API key anywhere in the repository.**

If your room's connection is unreliable, set `config.OFFLINE = True` and confirm
the whole notebook still runs. Only the generation cells need the network.
